# IF Fisher Diagonal Analysis (Standalone)

This notebook reuses the Fisher analysis logic from
`05_fisher_merging_precision_weighted.ipynb` and applies it to a precomputed
single artifact:

- `/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/fisher_diagonal/fisher_diag_if.pt`

## What this notebook runs

1. **Fisher diagonal concentration diagnostics** (sampling, Lorenz curve, Gini, top-k mass share)
2. **Fisher dominant layer/module/head diagnostics** (layer-level and head-level localization)

The code is intentionally kept close to notebook `05` so results remain directly comparable.


In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Mapping, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

# `transformers` may be unavailable in minimal environments.
# We keep an explicit fallback path to `config.json` so head analysis remains runnable.
try:
    from transformers import AutoConfig
except Exception:
    AutoConfig = None


@dataclass(frozen=True)
class FisherAnalysisSpec:
    """Specification for a single Fisher-analysis target.

    Args:
        name: Human-readable key used in table/plot labels.
        model_path: HF checkpoint path used to resolve architecture metadata.
        fisher_path: Path to serialized Fisher-diagonal `.pt` mapping.
    """

    name: str
    model_path: Path
    fisher_path: Path


# Fixed paths requested for standalone IF Fisher diagnostics.
IF_FISHER_SPEC = FisherAnalysisSpec(
    name='if',
    model_path=Path('/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface'),
    fisher_path=Path('/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/fisher_diagonal/fisher_diag_if.pt'),
)

TASK_SPECS = (IF_FISHER_SPEC,)
OUTPUT_ROOT = Path('merging_analysis/artifacts/fisher_diag_if_analysis')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Reproducible sampling seed used by concentration diagnostics.
RANDOM_SEED = 7

print('Configured task specs:')
for spec in TASK_SPECS:
    print(f"- {spec.name}: model={spec.model_path}, fisher={spec.fisher_path}")
print(f'Output root: {OUTPUT_ROOT}')


In [ ]:
# Fisher diagonal concentration diagnostics
# This cell helps answer whether Fisher scale is concentrated in a small subset of
# parameters (sparse) or spread relatively uniformly across the whole model.

from typing import Any, Dict, Mapping, Tuple


def load_fisher_dictionary(task_name: str, fisher_path: Path) -> Mapping[str, torch.Tensor]:
    """Load one task's Fisher diagonal dictionary from disk.

    Args:
        task_name: Short task key used for readable error messages.
        fisher_path: Path to the serialized Fisher tensor dictionary (`.pt`).

    Returns:
        Parameter-name keyed Fisher diagonal dictionary on CPU.

    Raises:
        FileNotFoundError: If the expected Fisher file is missing.
        TypeError: If the deserialized object is not a mapping.
    """

    if not fisher_path.exists():
        raise FileNotFoundError(f"Fisher file for task '{task_name}' was not found: {fisher_path}")

    fisher_obj = torch.load(fisher_path, map_location='cpu')
    if not isinstance(fisher_obj, Mapping):
        raise TypeError(
            f"Fisher file for task '{task_name}' must contain a mapping, got {type(fisher_obj)}"
        )

    return fisher_obj


def collect_tensor_mass_stats(
    fisher_tensors: Mapping[str, torch.Tensor],
) -> Tuple[pd.DataFrame, int, float]:
    """Collect per-parameter Fisher mass statistics.

    Why this exists:
        Tensor-level mass share reveals whether only a few named parameters/layers
        dominate the total Fisher mass.

    Args:
        fisher_tensors: Mapping from parameter name to Fisher diagonal tensor.

    Returns:
        Tuple of:
        - DataFrame with per-parameter stats (`numel`, `mass`, `mean`, `max`, `mass_share`)
        - Total element count across all tensors
        - Total Fisher mass across all tensors
    """

    rows = []
    total_numel = 0
    total_mass = 0.0

    for parameter_name, fisher_tensor in fisher_tensors.items():
        # We use absolute value defensively in case tiny negative values appear from
        # numerical noise; Fisher diagonals should be non-negative in theory.
        flat_values = fisher_tensor.detach().to(torch.float32).abs().reshape(-1)

        numel = int(flat_values.numel())
        mass = float(flat_values.sum().item())
        mean_value = float(mass / max(numel, 1))
        max_value = float(flat_values.max().item()) if numel > 0 else 0.0

        rows.append(
            {
                'parameter_name': parameter_name,
                'numel': numel,
                'mass': mass,
                'mean': mean_value,
                'max': max_value,
            }
        )

        total_numel += numel
        total_mass += mass

    tensor_stats_df = pd.DataFrame(rows)
    if total_mass > 0.0 and not tensor_stats_df.empty:
        tensor_stats_df['mass_share'] = tensor_stats_df['mass'] / total_mass
    else:
        tensor_stats_df['mass_share'] = 0.0

    return tensor_stats_df, total_numel, total_mass


def sample_fisher_values(
    fisher_tensors: Mapping[str, torch.Tensor],
    total_numel: int,
    sample_size: int,
    seed: int,
) -> np.ndarray:
    """Uniformly sample Fisher elements without materializing one giant vector.

    Why this exists:
        Full flattening across all model parameters is memory-heavy for large LMs.
        This two-pass index strategy gives an unbiased global sample with low memory.

    Args:
        fisher_tensors: Mapping from parameter name to Fisher tensor.
        total_numel: Total number of scalar elements across all tensors.
        sample_size: Number of scalar Fisher values to sample.
        seed: RNG seed for reproducibility.

    Returns:
        1D NumPy array of sampled non-negative Fisher values.
    """

    if total_numel <= 0 or sample_size <= 0:
        return np.zeros(0, dtype=np.float32)

    effective_sample_size = int(min(sample_size, total_numel))
    rng = np.random.default_rng(seed=seed)

    # Sampling with replacement keeps memory predictable even when total_numel is huge.
    sampled_global_indices = np.sort(
        rng.integers(low=0, high=total_numel, size=effective_sample_size, dtype=np.int64)
    )

    sampled_values = np.empty(effective_sample_size, dtype=np.float32)
    write_cursor = 0
    tensor_offset = 0

    for fisher_tensor in fisher_tensors.values():
        flat_values = fisher_tensor.detach().to(torch.float32).abs().reshape(-1)
        tensor_numel = int(flat_values.numel())

        # Locate sampled global indices that fall into the current tensor range.
        left = np.searchsorted(sampled_global_indices, tensor_offset, side='left')
        right = np.searchsorted(sampled_global_indices, tensor_offset + tensor_numel, side='left')

        if right > left:
            local_indices = sampled_global_indices[left:right] - tensor_offset
            local_index_tensor = torch.from_numpy(local_indices.astype(np.int64))
            local_values = flat_values.index_select(dim=0, index=local_index_tensor)

            next_cursor = write_cursor + (right - left)
            sampled_values[write_cursor:next_cursor] = local_values.cpu().numpy()
            write_cursor = next_cursor

        tensor_offset += tensor_numel

    if write_cursor != effective_sample_size:
        raise RuntimeError(
            f"Sampling bookkeeping mismatch: expected {effective_sample_size}, got {write_cursor}"
        )

    return sampled_values


def compute_lorenz_curve(values: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Compute Lorenz curve points for non-negative values.

    Args:
        values: 1D array of Fisher values.

    Returns:
        Tuple of arrays (`population_share`, `mass_share`).
    """

    if values.size == 0:
        return np.array([0.0, 1.0]), np.array([0.0, 1.0])

    clipped = np.clip(values.astype(np.float64), a_min=0.0, a_max=None)
    sorted_values = np.sort(clipped)
    cumulative_mass = np.cumsum(sorted_values)

    total_mass = float(cumulative_mass[-1])
    if total_mass <= 0.0:
        return np.array([0.0, 1.0]), np.array([0.0, 1.0])

    mass_share = np.concatenate(([0.0], cumulative_mass / total_mass))
    population_share = np.linspace(0.0, 1.0, mass_share.size)
    return population_share, mass_share


def compute_gini_from_lorenz(population_share: np.ndarray, mass_share: np.ndarray) -> float:
    """Compute Gini coefficient from Lorenz curve points.

    Args:
        population_share: X-axis values of Lorenz curve.
        mass_share: Y-axis values of Lorenz curve.

    Returns:
        Gini coefficient in [0, 1], where larger means more concentration.
    """

    area_under_curve = float(np.trapz(mass_share, population_share))
    gini = 1.0 - 2.0 * area_under_curve
    return float(np.clip(gini, 0.0, 1.0))


def compute_top_mass_share(values: np.ndarray, top_fraction: float) -> float:
    """Compute the mass captured by the top fraction of sampled values.

    Args:
        values: Sampled Fisher values.
        top_fraction: Fraction in (0, 1], e.g. 0.01 for top 1%.

    Returns:
        Fraction of total sampled mass carried by top-ranked elements.
    """

    if values.size == 0:
        return 0.0

    clipped = np.clip(values.astype(np.float64), a_min=0.0, a_max=None)
    total_mass = float(clipped.sum())
    if total_mass <= 0.0:
        return 0.0

    top_count = int(max(1, np.ceil(clipped.size * float(top_fraction))))
    top_values = np.partition(clipped, clipped.size - top_count)[-top_count:]
    return float(top_values.sum() / total_mass)


SAMPLE_SIZE_PER_TASK = 500_000
TOP_FRACTIONS = (0.001, 0.01, 0.05, 0.10)
EPSILON_FOR_LOG = 1e-30


task_diagnostics: Dict[str, Dict[str, Any]] = {}

for task_spec in TASK_SPECS:
    task_name = task_spec.name
    fisher_path = task_spec.fisher_path

    fisher_tensors = load_fisher_dictionary(task_name=task_name, fisher_path=fisher_path)
    tensor_stats_df, total_numel, total_mass = collect_tensor_mass_stats(fisher_tensors=fisher_tensors)
    sampled_values = sample_fisher_values(
        fisher_tensors=fisher_tensors,
        total_numel=total_numel,
        sample_size=SAMPLE_SIZE_PER_TASK,
        seed=RANDOM_SEED,
    )

    population_share, mass_share = compute_lorenz_curve(sampled_values)

    task_diagnostics[task_name] = {
        'tensor_stats_df': tensor_stats_df,
        'total_numel': total_numel,
        'total_mass': total_mass,
        'sampled_values': sampled_values,
        'population_share': population_share,
        'mass_share': mass_share,
    }

# Build a compact numeric summary so sparsity/uniformity can be compared directly.
summary_rows = []
for task_name, diag in task_diagnostics.items():
    sampled_values = diag['sampled_values']
    tensor_stats_df = diag['tensor_stats_df']

    top_tensor_share = (
        float(tensor_stats_df['mass_share'].max()) if not tensor_stats_df.empty else 0.0
    )

    row = {
        'task': task_name,
        'total_numel': int(diag['total_numel']),
        'total_mass': float(diag['total_mass']),
        'sample_size': int(sampled_values.size),
        'sampled_nonzero_fraction': float((sampled_values > 0).mean()) if sampled_values.size > 0 else 0.0,
        'gini_sampled': compute_gini_from_lorenz(diag['population_share'], diag['mass_share']),
        'top_tensor_mass_share': top_tensor_share,
    }

    for fraction in TOP_FRACTIONS:
        key = f"top_{fraction * 100:.1f}pct_mass_share".replace('.', '_')
        row[key] = compute_top_mass_share(sampled_values, top_fraction=fraction)

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values('task').reset_index(drop=True)
print('Interpretation hint: higher Gini / higher top-x% mass share means stronger concentration (sparser Fisher scale).')
display(summary_df)

# 2x2 panel to compare element-level and tensor-level concentration patterns.
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
ax_hist, ax_lorenz, ax_topk, ax_tensor_cum = axes.flatten()

for task_name, diag in task_diagnostics.items():
    sampled_values = diag['sampled_values']
    log_values = np.log10(np.clip(sampled_values, a_min=EPSILON_FOR_LOG, a_max=None))

    ax_hist.hist(
        log_values,
        bins=120,
        density=True,
        alpha=0.45,
        label=task_name,
    )

ax_hist.set_title('Sampled Fisher distribution (log10 scale)')
ax_hist.set_xlabel('log10(Fisher + epsilon)')
ax_hist.set_ylabel('Density')
ax_hist.legend()

for task_name, diag in task_diagnostics.items():
    ax_lorenz.plot(diag['population_share'], diag['mass_share'], linewidth=2.0, label=task_name)

ax_lorenz.plot([0.0, 1.0], [0.0, 1.0], 'k--', linewidth=1.0, label='Uniform baseline')
ax_lorenz.set_title('Lorenz curve of sampled Fisher mass')
ax_lorenz.set_xlabel('Fraction of parameters (sampled)')
ax_lorenz.set_ylabel('Cumulative Fisher mass fraction')
ax_lorenz.legend()

fraction_labels = [f"top {fraction * 100:.1f}%" for fraction in TOP_FRACTIONS]
bar_positions = np.arange(len(fraction_labels), dtype=np.float64)
task_names = list(task_diagnostics.keys())
bar_width = 0.8 / max(len(task_names), 1)

for task_index, task_name in enumerate(task_names):
    sampled_values = task_diagnostics[task_name]['sampled_values']
    heights = [compute_top_mass_share(sampled_values, top_fraction=fraction) for fraction in TOP_FRACTIONS]

    offset = (task_index - (len(task_names) - 1) / 2.0) * bar_width
    ax_topk.bar(bar_positions + offset, heights, width=bar_width, label=task_name)

ax_topk.set_xticks(bar_positions)
ax_topk.set_xticklabels(fraction_labels, rotation=0)
ax_topk.set_ylim(0.0, 1.0)
ax_topk.set_title('Mass captured by top-ranked sampled elements')
ax_topk.set_ylabel('Fisher mass fraction')
ax_topk.legend()

for task_name, diag in task_diagnostics.items():
    tensor_stats_df = diag['tensor_stats_df'].sort_values('mass_share', ascending=False)
    if tensor_stats_df.empty:
        continue

    cumulative_mass = tensor_stats_df['mass_share'].cumsum().to_numpy()
    tensor_fraction = np.arange(1, cumulative_mass.size + 1, dtype=np.float64) / cumulative_mass.size
    ax_tensor_cum.plot(tensor_fraction, cumulative_mass, linewidth=2.0, label=task_name)

ax_tensor_cum.plot([0.0, 1.0], [0.0, 1.0], 'k--', linewidth=1.0, label='Uniform baseline')
ax_tensor_cum.set_title('Cumulative mass over parameter tensors')
ax_tensor_cum.set_xlabel('Fraction of parameter tensors (sorted by mass)')
ax_tensor_cum.set_ylabel('Cumulative Fisher mass fraction')
ax_tensor_cum.legend()

plt.tight_layout()
plt.show()

# Print top tensor contributors for quick qualitative inspection.
for task_name, diag in task_diagnostics.items():
    print(f"\nTop 15 parameter tensors by Fisher mass share - task={task_name}")
    display(
        diag['tensor_stats_df']
        .sort_values('mass_share', ascending=False)
        .head(15)
        [['parameter_name', 'numel', 'mass_share', 'mean', 'max']]
        .reset_index(drop=True)
    )

# Persist tabular outputs so the notebook can be reused in CLI/plot pipelines.
for task_name, diag in task_diagnostics.items():
    task_out_dir = OUTPUT_ROOT / task_name / 'concentration'
    task_out_dir.mkdir(parents=True, exist_ok=True)

    diag['tensor_stats_df'].to_csv(task_out_dir / 'tensor_mass_stats.csv', index=False)

    summary_payload = {
        'task': task_name,
        'total_numel': int(diag['total_numel']),
        'total_mass': float(diag['total_mass']),
        'sample_size': int(diag['sampled_values'].size),
        'sampled_nonzero_fraction': float((diag['sampled_values'] > 0).mean()) if diag['sampled_values'].size > 0 else 0.0,
        'gini_sampled': compute_gini_from_lorenz(diag['population_share'], diag['mass_share']),
    }
    for fraction in TOP_FRACTIONS:
        key = f"top_{fraction * 100:.1f}pct_mass_share".replace('.', '_')
        summary_payload[key] = compute_top_mass_share(diag['sampled_values'], top_fraction=fraction)

    (task_out_dir / 'summary.json').write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False))

print(f'Saved concentration artifacts under: {OUTPUT_ROOT}')


In [ ]:
# Fisher dominant layer/head analysis
# This cell localizes which layers, module groups, and attention heads dominate
# Fisher mass when global concentration metrics (e.g., high Gini) are extreme.

import re
from typing import Mapping, Optional, Tuple


LAYER_PATTERN = re.compile(r"^model\.layers\.(\d+)\.")
ATTN_QKV_WEIGHT_PATTERN = re.compile(
    r"^model\.layers\.(\d+)\.self_attn\.(q_proj|k_proj|v_proj)\.weight$"
)


def load_fisher_dictionary_for_task(task_name: str, fisher_path: Path) -> Mapping[str, torch.Tensor]:
    """Load one task Fisher dictionary from disk.

    Args:
        task_name: Task key used for diagnostics.
        fisher_path: Path to serialized Fisher-diagonal `.pt` mapping.

    Returns:
        Mapping from parameter name to Fisher tensor.

    Raises:
        FileNotFoundError: If Fisher artifact is missing.
        TypeError: If loaded object is not a mapping.
    """

    if not fisher_path.exists():
        raise FileNotFoundError(f"Missing Fisher file for task '{task_name}': {fisher_path}")

    fisher_obj = torch.load(fisher_path, map_location='cpu')
    if not isinstance(fisher_obj, Mapping):
        raise TypeError(
            f"Fisher object for task '{task_name}' must be a mapping, got {type(fisher_obj)}"
        )

    return fisher_obj


def extract_layer_index(parameter_name: str) -> Optional[int]:
    """Extract transformer layer index from parameter name.

    Args:
        parameter_name: Named parameter key in Hugging Face model format.

    Returns:
        Integer layer index when key belongs to `model.layers.{idx}.*`, else `None`.
    """

    match = LAYER_PATTERN.match(parameter_name)
    if match is None:
        return None
    return int(match.group(1))


def classify_module_group(parameter_name: str) -> str:
    """Classify parameter into interpretable module groups.

    Why this exists:
        Group-level aggregation (attn/MLP/norm/embedding) makes it easy to see
        whether Fisher concentration is dominated by a specific subsystem.

    Args:
        parameter_name: Named parameter key.

    Returns:
        Module-group label string.
    """

    if parameter_name.startswith('model.embed_tokens.'):
        return 'embed_tokens'
    if parameter_name.startswith('lm_head.'):
        return 'lm_head'
    if parameter_name.startswith('model.norm.'):
        return 'final_norm'

    if '.self_attn.q_proj.' in parameter_name:
        return 'attn_q_proj'
    if '.self_attn.k_proj.' in parameter_name:
        return 'attn_k_proj'
    if '.self_attn.v_proj.' in parameter_name:
        return 'attn_v_proj'
    if '.self_attn.o_proj.' in parameter_name:
        return 'attn_o_proj'

    if '.mlp.gate_proj.' in parameter_name:
        return 'mlp_gate_proj'
    if '.mlp.up_proj.' in parameter_name:
        return 'mlp_up_proj'
    if '.mlp.down_proj.' in parameter_name:
        return 'mlp_down_proj'

    if '.input_layernorm.' in parameter_name:
        return 'input_layernorm'
    if '.post_attention_layernorm.' in parameter_name:
        return 'post_attention_layernorm'

    return 'other'


def format_layer_label(layer_index: int) -> str:
    """Format integer layer index into compact display label."""

    if layer_index < 0:
        return 'non_transformer'
    return f"L{layer_index:02d}"


def collect_parameter_mass_dataframe(
    fisher_tensors: Mapping[str, torch.Tensor],
) -> Tuple[pd.DataFrame, float]:
    """Build per-parameter Fisher mass table.

    Args:
        fisher_tensors: Fisher diagonal tensors keyed by parameter name.

    Returns:
        Tuple of:
        - DataFrame with per-parameter metadata and Fisher mass statistics.
        - Total Fisher mass across all parameters.
    """

    rows = []
    total_mass = 0.0

    for parameter_name, fisher_tensor in fisher_tensors.items():
        # Absolute value is used defensively against tiny numerical negatives.
        fisher_abs = fisher_tensor.detach().to(torch.float32).abs()

        parameter_mass = float(fisher_abs.sum().item())
        parameter_numel = int(fisher_abs.numel())

        layer_index = extract_layer_index(parameter_name)
        normalized_layer_index = int(layer_index) if layer_index is not None else -1

        rows.append(
            {
                'parameter_name': parameter_name,
                'layer_index': normalized_layer_index,
                'layer_label': format_layer_label(normalized_layer_index),
                'module_group': classify_module_group(parameter_name),
                'numel': parameter_numel,
                'mass': parameter_mass,
                'mean': float(parameter_mass / max(parameter_numel, 1)),
                'max': float(fisher_abs.max().item()) if parameter_numel > 0 else 0.0,
            }
        )

        total_mass += parameter_mass

    parameter_df = pd.DataFrame(rows)
    if parameter_df.empty:
        return parameter_df, total_mass

    if total_mass > 0.0:
        parameter_df['mass_share_global'] = parameter_df['mass'] / total_mass
    else:
        parameter_df['mass_share_global'] = 0.0

    return parameter_df, total_mass


def build_layer_module_summaries(parameter_df: pd.DataFrame, total_mass: float) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Aggregate parameter Fisher mass to layer and layer-module levels.

    Args:
        parameter_df: Per-parameter Fisher mass table.
        total_mass: Total Fisher mass across all parameters.

    Returns:
        Tuple of:
        - Layer-level summary table.
        - Layer+module-group summary table.
    """

    if parameter_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    layer_df = (
        parameter_df
        .groupby(['layer_index', 'layer_label'], as_index=False)
        .agg(mass=('mass', 'sum'), numel=('numel', 'sum'))
    )
    layer_df['mass_share_global'] = 0.0 if total_mass <= 0.0 else (layer_df['mass'] / total_mass)
    layer_df['mean'] = layer_df['mass'] / layer_df['numel'].clip(lower=1)
    layer_df = layer_df.sort_values('mass_share_global', ascending=False).reset_index(drop=True)

    layer_module_df = (
        parameter_df
        .groupby(['layer_index', 'layer_label', 'module_group'], as_index=False)
        .agg(mass=('mass', 'sum'), numel=('numel', 'sum'))
    )
    layer_module_df['mass_share_global'] = (
        0.0 if total_mass <= 0.0 else (layer_module_df['mass'] / total_mass)
    )
    layer_module_df['mean'] = layer_module_df['mass'] / layer_module_df['numel'].clip(lower=1)
    layer_module_df = layer_module_df.sort_values('mass_share_global', ascending=False).reset_index(drop=True)

    return layer_df, layer_module_df


def resolve_num_attention_heads(reference_model_path: Path) -> Optional[int]:
    """Read number of attention heads from model config.

    This function follows the original `05` logic (AutoConfig) and adds one safe
    fallback path to local `config.json` parsing for environments where
    `transformers` is not installed.

    Args:
        reference_model_path: Local model path with HF config.

    Returns:
        `num_attention_heads` when available, otherwise `None`.
    """

    if AutoConfig is not None:
        try:
            config = AutoConfig.from_pretrained(str(reference_model_path), trust_remote_code=True)
            num_heads = getattr(config, 'num_attention_heads', None)
            if num_heads is not None:
                return int(num_heads)
        except Exception as exc:
            print(f"[warn] Failed to load AutoConfig from {reference_model_path}: {exc}")

    # Fallback: parse raw config.json directly.
    config_json_path = reference_model_path / 'config.json'
    if not config_json_path.exists():
        print(f"[warn] config.json is missing: {config_json_path}")
        return None

    try:
        config_dict = json.loads(config_json_path.read_text())
    except Exception as exc:
        print(f"[warn] Failed to parse config.json at {config_json_path}: {exc}")
        return None

    num_heads = config_dict.get('num_attention_heads', None)
    if num_heads is None:
        print('[warn] num_attention_heads is missing in model config; head analysis is skipped.')
        return None

    return int(num_heads)


def collect_attention_head_mass(
    fisher_tensors: Mapping[str, torch.Tensor],
    num_attention_heads: int,
    total_mass: float,
) -> pd.DataFrame:
    """Compute per-layer per-head Fisher mass from Q/K/V projection weights.

    Important note:
        This decomposition assumes Q/K/V projection output channels are grouped
        contiguously by head, which matches standard transformer implementations.

    Args:
        fisher_tensors: Fisher diagonal tensors keyed by parameter name.
        num_attention_heads: Number of attention heads from model config.
        total_mass: Total Fisher mass across all parameters.

    Returns:
        DataFrame with per-head Fisher mass and global mass share.
    """

    rows = []

    for parameter_name, fisher_tensor in fisher_tensors.items():
        match = ATTN_QKV_WEIGHT_PATTERN.match(parameter_name)
        if match is None:
            continue

        layer_index = int(match.group(1))
        projection_name = str(match.group(2))

        fisher_abs = fisher_tensor.detach().to(torch.float32).abs()
        if fisher_abs.ndim != 2:
            # Q/K/V projection weights are expected to be 2D linear matrices.
            continue

        out_features = int(fisher_abs.shape[0])
        in_features = int(fisher_abs.shape[1])
        if out_features % int(num_attention_heads) != 0:
            # Skip if head partitioning is ambiguous for this tensor shape.
            continue

        head_dim = out_features // int(num_attention_heads)
        per_head_mass = fisher_abs.reshape(num_attention_heads, head_dim, in_features).sum(dim=(1, 2))

        for head_index, mass_value in enumerate(per_head_mass.tolist()):
            rows.append(
                {
                    'layer_index': layer_index,
                    'layer_label': format_layer_label(layer_index),
                    'projection': projection_name,
                    'head_index': int(head_index),
                    'mass': float(mass_value),
                }
            )

    head_projection_df = pd.DataFrame(rows)
    if head_projection_df.empty:
        return head_projection_df

    # Aggregate across q/k/v so each row becomes a single head in one layer.
    head_df = (
        head_projection_df
        .groupby(['layer_index', 'layer_label', 'head_index'], as_index=False)
        .agg(mass=('mass', 'sum'))
    )

    attention_total_mass = float(head_df['mass'].sum())
    head_df['global_mass_share'] = 0.0 if total_mass <= 0.0 else (head_df['mass'] / total_mass)
    head_df['attention_mass_share'] = (
        0.0 if attention_total_mass <= 0.0 else (head_df['mass'] / attention_total_mass)
    )

    head_df = head_df.sort_values('global_mass_share', ascending=False).reset_index(drop=True)
    return head_df


TOPK_LAYER_ROWS = 20
TOPK_LAYER_MODULE_ROWS = 30
TOPK_HEAD_ROWS = 40

# Use task model config as attention-head reference.
reference_model_path = TASK_SPECS[0].model_path
num_attention_heads = resolve_num_attention_heads(reference_model_path=reference_model_path)
print(f"Attention head count from config: {num_attention_heads}")

for task_spec in TASK_SPECS:
    task_name = task_spec.name
    fisher_path = task_spec.fisher_path

    print(f"\n=== Fisher dominant-parameter diagnostics: task={task_name} ===")

    fisher_tensors = load_fisher_dictionary_for_task(task_name=task_name, fisher_path=fisher_path)
    parameter_df, total_mass = collect_parameter_mass_dataframe(fisher_tensors=fisher_tensors)

    if parameter_df.empty:
        print(f"No Fisher tensors found for task={task_name}; skipping.")
        continue

    layer_df, layer_module_df = build_layer_module_summaries(parameter_df=parameter_df, total_mass=total_mass)

    print(f"Total Fisher mass: {total_mass:.6e}")
    print(f"Num parameter tensors: {len(parameter_df)}")

    print("\nTop layers by global Fisher mass share:")
    display(layer_df.head(TOPK_LAYER_ROWS)[['layer_label', 'mass_share_global', 'mass', 'numel', 'mean']])

    print("\nTop (layer, module_group) by global Fisher mass share:")
    display(
        layer_module_df.head(TOPK_LAYER_MODULE_ROWS)[
            ['layer_label', 'module_group', 'mass_share_global', 'mass', 'numel', 'mean']
        ]
    )

    print("\nTop parameter tensors by global Fisher mass share:")
    display(
        parameter_df
        .sort_values('mass_share_global', ascending=False)
        .head(20)[['parameter_name', 'layer_label', 'module_group', 'mass_share_global', 'mean', 'max']]
        .reset_index(drop=True)
    )

    head_df = pd.DataFrame()
    if num_attention_heads is not None and num_attention_heads > 0:
        head_df = collect_attention_head_mass(
            fisher_tensors=fisher_tensors,
            num_attention_heads=num_attention_heads,
            total_mass=total_mass,
        )

    if not head_df.empty:
        print("\nTop attention heads by global Fisher mass share (aggregated over q/k/v):")
        display(
            head_df.head(TOPK_HEAD_ROWS)[
                ['layer_label', 'head_index', 'global_mass_share', 'attention_mass_share', 'mass']
            ]
        )
    else:
        print("\nAttention head analysis unavailable (config missing or no q/k/v matrices matched).")

    # Visualization panel: top layers + layer-module heatmap + head heatmap.
    fig, axes = plt.subplots(1, 3, figsize=(24, 6))

    # Panel 1: top layers bar chart.
    top_layer_plot_df = layer_df.head(TOPK_LAYER_ROWS).sort_values('mass_share_global', ascending=True)
    axes[0].barh(top_layer_plot_df['layer_label'], top_layer_plot_df['mass_share_global'])
    axes[0].set_title(f"Top layers by Fisher mass share ({task_name})")
    axes[0].set_xlabel('Global Fisher mass share')
    axes[0].set_ylabel('Layer')

    # Panel 2: layer x module-group heatmap.
    layer_module_pivot = (
        layer_module_df
        .pivot(index='layer_label', columns='module_group', values='mass_share_global')
        .fillna(0.0)
    )

    # Sort layer labels numerically for readability, keeping non-transformer at the end.
    sorted_layer_labels = sorted(
        layer_module_pivot.index.tolist(),
        key=lambda label: (9999 if label == 'non_transformer' else int(label[1:])),
    )
    layer_module_pivot = layer_module_pivot.loc[sorted_layer_labels]

    heatmap_module = axes[1].imshow(layer_module_pivot.to_numpy(), aspect='auto', cmap='magma')
    axes[1].set_title(f"Layer x module Fisher mass share ({task_name})")
    axes[1].set_xlabel('Module group')
    axes[1].set_ylabel('Layer')
    axes[1].set_xticks(range(len(layer_module_pivot.columns)))
    axes[1].set_xticklabels(layer_module_pivot.columns.tolist(), rotation=45, ha='right')
    axes[1].set_yticks(range(len(layer_module_pivot.index)))
    axes[1].set_yticklabels(layer_module_pivot.index.tolist())
    fig.colorbar(heatmap_module, ax=axes[1], fraction=0.046, pad=0.04)

    # Panel 3: layer x head heatmap for q/k/v projections.
    if not head_df.empty:
        head_pivot = head_df.pivot(index='layer_label', columns='head_index', values='global_mass_share').fillna(0.0)
        sorted_head_layers = sorted(
            head_pivot.index.tolist(),
            key=lambda label: (9999 if label == 'non_transformer' else int(label[1:])),
        )
        head_pivot = head_pivot.loc[sorted_head_layers]

        heatmap_head = axes[2].imshow(head_pivot.to_numpy(), aspect='auto', cmap='viridis')
        axes[2].set_title(f"Attention head Fisher mass share ({task_name})")
        axes[2].set_xlabel('Head index')
        axes[2].set_ylabel('Layer')
        axes[2].set_xticks(range(len(head_pivot.columns)))
        axes[2].set_xticklabels(head_pivot.columns.tolist())
        axes[2].set_yticks(range(len(head_pivot.index)))
        axes[2].set_yticklabels(head_pivot.index.tolist())
        fig.colorbar(heatmap_head, ax=axes[2], fraction=0.046, pad=0.04)
    else:
        axes[2].axis('off')
        axes[2].text(
            0.5,
            0.5,
            'Head-level analysis unavailable',
            ha='center',
            va='center',
            fontsize=12,
        )

    plt.tight_layout()
    plt.show()

    # Persist detailed tables for downstream checks and reproducibility.
    task_out_dir = OUTPUT_ROOT / task_name / 'dominant_layer_head'
    task_out_dir.mkdir(parents=True, exist_ok=True)

    layer_df.to_csv(task_out_dir / 'layer_df.csv', index=False)
    layer_module_df.to_csv(task_out_dir / 'layer_module_df.csv', index=False)
    parameter_df.sort_values('mass_share_global', ascending=False).to_csv(
        task_out_dir / 'parameter_df_sorted.csv',
        index=False,
    )
    if not head_df.empty:
        head_df.to_csv(task_out_dir / 'head_df.csv', index=False)

    summary_payload = {
        'task': task_name,
        'num_attention_heads': int(num_attention_heads) if num_attention_heads is not None else None,
        'total_mass': float(total_mass),
        'num_parameter_tensors': int(len(parameter_df)),
    }
    (task_out_dir / 'summary.json').write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False))

print(f'Saved dominant layer/head artifacts under: {OUTPUT_ROOT}')


## Output Artifacts

After running all cells, analysis outputs are stored under:

- `merging_analysis/artifacts/fisher_diag_if_analysis/if/concentration/`
- `merging_analysis/artifacts/fisher_diag_if_analysis/if/dominant_layer_head/`
